# Algorithmic VWAP Sigma Engine

Clean research notebook for the VWAP sigma-band strategy.

This notebook is used to:
- calculate VWAP and sigma probability bands
- detect continuation and mean-reversion setups
- route/filter raw signals
- simulate trades
- compare strategy results
- audit skipped opportunities
- prepare the final logic for a future live MT5 runner

This is not the live runner yet.

## Table of Contents

1. Imports and paths
2. Strategy control panel
3. Dataset configuration
4. VWAP and sigma band engine
5. Continuation signal engine
6. Mean-reversion signal engine
7. Router and strategy filter
8. Trade simulation and risk engine
9. Experiment runner
10. Quick results
11. Core charts
12. Risk diagnostics
13. Audits and detailed diagnostics
14. Experiment logging
15. Future live runner notes

## Imports and paths

This section contains the notebook imports, project-root detection, and output paths.

In [5]:
from pathlib import Path
from pprint import pprint
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start_path: Path | None = None) -> Path:
    """
    Find the project root by walking upward until the expected repository
    structure is found.

    This works whether the project was cloned with Git or downloaded as a ZIP,
    and whether the notebook is run from the project root or from notebooks/.
    """
    if start_path is None:
        start_path = Path.cwd()

    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        has_project_structure = (
            (path / "src").is_dir()
            and (path / "data").is_dir()
            and (path / "notebooks").is_dir()
        )

        has_repo_marker = (
            (path / ".git").exists()
            or (path / "README.md").exists()
            or (path / "requirements.txt").exists()
        )

        if has_project_structure and has_repo_marker:
            return path

    raise FileNotFoundError(
        "Could not find the project root. Make sure this notebook is being run "
        "from inside the VWAP-probability-band-engine project folder."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
HISTORICAL_DATA_DIR = DATA_DIR / "historical"
SRC_DIR = PROJECT_ROOT / "src"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_DIR / "algorithmic_vwap_sigma_engine"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Detected project root:", PROJECT_ROOT)
print("Historical data folder:", HISTORICAL_DATA_DIR)
print("Source folder:", SRC_DIR)
print("Output folder:", OUTPUT_DIR)

Current working directory: c:\GitHub Projects\VWAP-probability-band-engine\notebooks
Detected project root: C:\GitHub Projects\VWAP-probability-band-engine
Historical data folder: C:\GitHub Projects\VWAP-probability-band-engine\data\historical
Source folder: C:\GitHub Projects\VWAP-probability-band-engine\src
Output folder: C:\GitHub Projects\VWAP-probability-band-engine\artifacts\algorithmic_vwap_sigma_engine


## Strategy control panel

This section controls the strategy behaviour from one place.

The control panel is separated by role:
- strategy family switches
- continuation entry-type switches
- router and strategy-filter controls
- setup-specific quality filters
- future router-blocked override controls
- execution and risk controls
- reporting and audit controls

The goal is to make it clear which stage each setting affects.

In [6]:
# ============================================================
# 1. MAIN STRATEGY SWITCHES
# ============================================================
# Purpose:
#   Turn major strategy families on or off.
#
# Scope:
#   These are top-level switches. If a family is disabled here,
#   that family should not create trades.
#
# Default:
#   Continuation is enabled. Mean reversion and add-ons are disabled
#   until proven stable.
#
# Risk:
#   Enabling extra strategy families can change trade frequency,
#   drawdown, and attribution.

ENABLE_CONTINUATION = True
ENABLE_MEAN_REVERSION = False
ENABLE_PROTECTED_ADDONS = False


# ============================================================
# 2. CONTINUATION ENTRY TYPES
# ============================================================
# Purpose:
#   Control which continuation setup families can create raw signals.
#
# Scope:
#   These switches affect raw continuation setup creation only.
#
# Default:
#   The current primary continuation profile keeps the main continuation
#   families available.
#
# Risk:
#   Disabling a setup family can hide valid opportunity types.
#   Enabling too many weak setup families without filtering can increase noise.

ENABLE_S_TIER = True
ENABLE_DYNAMIC_S_TIER = True
ENABLE_A_TIER = True
ENABLE_DELAYED_PULLBACK = True


# ============================================================
# 3. ROUTER / STRATEGY FILTER
# ============================================================
# Purpose:
#   Decide whether raw setups fit the current market regime.
#
# Scope:
#   This is a permission layer after raw signal creation.
#   It should not be confused with raw setup detection.
#
# Default:
#   Enabled because it blocks many weak-regime signals.
#
# Risk:
#   Turning this off can allow many low-quality trades.
#   Leaving it too strict may block valid A-tier continuation opportunities.

USE_STRATEGY_FILTER = True
STRATEGY_FILTER_MODE = "v4_dynamic_regime_selector"


# ============================================================
# 4. STRATEGY FILTER OVERRIDES
# ============================================================
# Purpose:
#   Future research controls for router-blocked setups.
#
# Scope:
#   These should only apply after the router blocks a raw setup.
#   They should not affect router-approved trades.
#
# Default:
#   OFF. They should not change current results.
#
# Risk:
#   If implemented carelessly, these can become the same as removing
#   the strategy filter. They must require extra quality confirmation.

ALLOW_BLOCKED_S_TIER_WITH_EXTRA_QUALITY = False
ALLOW_BLOCKED_DYNAMIC_S_TIER_WITH_EXTRA_QUALITY = False
ALLOW_BLOCKED_A_TIER_WITH_EXTRA_QUALITY = False
ALLOW_BLOCKED_DELAYED_PULLBACK_WITH_EXTRA_QUALITY = False


# ============================================================
# 5. SETUP-SPECIFIC QUALITY FILTERS
# ============================================================
# Purpose:
#   Extra quality checks after a raw setup exists.
#
# Scope:
#   These should clearly state whether they apply globally,
#   only to S-tier, only to A-tier, only to delayed pullbacks,
#   only to mean reversion, or only to future router-blocked overrides.
#
# Default:
#   Preserve the current primary continuation profile.
#
# Risk:
#   Turning on too many filters at once may double-filter trades
#   that the strategy router already approved.

USE_S_TIER_CLEAN_STATE_FILTER = True

USE_RED_SHIFT_FLOOR = False
USE_TREND_HEALTH_FILTER = True
USE_CANDLE_QUALITY_FILTER = True
USE_EXTENSION_FILTER = True


# ============================================================
# 6. RISK AND RUNNER MANAGEMENT
# ============================================================
# Purpose:
#   Define continuation trade risk, target, breakeven, and runner behaviour.
#
# Scope:
#   Applies to continuation trades unless a setup-specific exit model overrides it.
#
# Default:
#   Uses the current primary continuation runner profile.
#
# Risk:
#   Changing stop size, target, breakeven, or runner locks can materially
#   change all headline results.

SL_POINTS = 36.0
ORIGINAL_TP_POINTS = 72.0
BE_TRIGGER_POINTS = 36.0

RUNNER_MODE = "always"
RUNNER_TARGET_R = 7.0
USE_LEGACY_EARLY_RUNNER_LOCK = False

RUNNER_TRAIL_RULES_R = [
    {"trigger_r": 1.0, "lock_r": 0.0, "label": "BE"},
    {"trigger_r": 2.7, "lock_r": 2.0, "label": "TRAIL_LOCK_2R"},
    {"trigger_r": 3.7, "lock_r": 3.0, "label": "TRAIL_LOCK_3R"},
    {"trigger_r": 4.7, "lock_r": 4.0, "label": "TRAIL_LOCK_4R"},
    {"trigger_r": 5.7, "lock_r": 5.0, "label": "TRAIL_LOCK_5R"},
    {"trigger_r": 6.7, "lock_r": 6.0, "label": "TRAIL_LOCK_6R"},
]


# ============================================================
# 7. PROTECTED CONTINUATION ADD-ONS
# ============================================================
# Purpose:
#   Allow another continuation trade only when an existing continuation
#   trade is already protected.
#
# Scope:
#   Applies to continuation stacking logic, not mean reversion.
#
# Default:
#   OFF until proven stable.
#
# Risk:
#   If configured badly, this can create multiple full-risk trades.
#   The intended rule is no second fresh full-risk trade.

ENABLE_PROTECTED_ADDONS = False
ADDON_REQUIRES_EXISTING_TRADE_PROTECTED = True
ADDON_REQUIRES_SAME_DIRECTION = True
MAX_OPEN_CONTINUATION_TRADES = 2


# ============================================================
# 8. MEAN-REVERSION SETTINGS
# ============================================================
# Purpose:
#   Configure the optional mean-reversion module.
#
# Scope:
#   Mean reversion is separate from continuation.
#   It targets VWAP and should not use the continuation runner.
#
# Default:
#   Disabled because early tests over-traded.
#
# Risk:
#   Mean reversion can add many trades and increase drawdown if not filtered.

ENABLE_MEAN_REVERSION = False

MR_TARGET_MODE = "vwap"
MR_STOP_MODE = "structure"
MR_MAX_SL_POINTS = 14.0
MR_MIN_TARGET_RR = 2.0

ENABLE_MR_GREEN_BE = True
MR_BE_TRIGGER_MODE = "inner_green"
MR_BE_REQUIRES_CLOSE_BEYOND_GREEN = True


# ============================================================
# 9. DAILY RISK / SESSION CONTROLS
# ============================================================
# Purpose:
#   Control account/risk/session-level trade permissions.
#
# Scope:
#   These controls operate after signal selection.
#
# Default:
#   Preserve current research profile.
#
# Risk:
#   Tight risk/session settings can reduce trade count.
#   Loose settings can increase drawdown exposure.

MAX_DAILY_LOSS_R = -2
DAILY_PROFIT_CAP_PCT = 999.0
MAX_CONSECUTIVE_SL = 2
NO_NEW_TRADES_AFTER = "19:00"
SESSION_FILTER_ENABLED = False


# ============================================================
# 10. AUDITS / REPORTS / LOGGING
# ============================================================
# Purpose:
#   Explain what the model saw, selected, executed, or skipped.
#
# Scope:
#   These should not change trading logic.
#
# Default:
#   Enabled because this notebook is a research/control notebook.
#
# Risk:
#   These should not affect results. If toggling a report changes trades,
#   something is wired incorrectly.

ENABLE_RAW_SIGNAL_AUDIT = True
ENABLE_STRATEGY_FILTER_BLOCK_AUDIT = True
ENABLE_TRADE_ATTRIBUTION_REPORT = True
ENABLE_RISK_DIAGNOSTICS = True
ENABLE_ACCOUNT_SURVIVAL_REPORT = True
ENABLE_EXPERIMENT_LOGGING = True

In [7]:
# ============================================================
# INTERNAL CONFIG MAPPING
# ============================================================
# Purpose:
#   Map the readable control-panel settings above into the internal keys
#   expected by the migrated engine code.
#
# Scope:
#   This cell is compatibility glue. Future edits should normally happen
#   in the readable control panel above, not by scattering overrides later.
#
# Default:
#   Keeps the current primary continuation profile active.
#
# Risk:
#   Changing internal keys here can alter strategy behaviour globally.

CONFIG = {
    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "csv_file": "US100_cash_M1_NY_session_1y.csv",
    "dataset_name": "US100_cash_M1_NY_session_1y",

    # ------------------------------------------------------------------
    # Strategy identity
    # ------------------------------------------------------------------
    "strategy_version": "algorithmic_vwap_sigma_engine",
    "strategy_description": "Clean modular VWAP sigma-band research engine",

    # ------------------------------------------------------------------
    # Session handling
    # ------------------------------------------------------------------
    "session_timezone": "Europe/London",
    "no_new_trades_after": NO_NEW_TRADES_AFTER,

    # ------------------------------------------------------------------
    # Direction controls
    # ------------------------------------------------------------------
    "allow_longs": True,
    "allow_shorts": True,

    # ------------------------------------------------------------------
    # Strategy family switches
    # ------------------------------------------------------------------
    "enable_continuation": ENABLE_CONTINUATION,
    "enable_v5_mr_orange_failure_entry": ENABLE_MEAN_REVERSION,
    "enable_protected_continuation_addons": ENABLE_PROTECTED_ADDONS,

    # ------------------------------------------------------------------
    # Continuation entry-type switches
    # ------------------------------------------------------------------
    "enable_v1_s_tier": ENABLE_S_TIER,
    "enable_dynamic_s_tier_touch_diagnostics": ENABLE_DYNAMIC_S_TIER,
    "enable_v3_a_tier_second_close": ENABLE_A_TIER,
    "enable_a_tier_delayed_pullback_entry": ENABLE_DELAYED_PULLBACK,

    # ------------------------------------------------------------------
    # Router / strategy-filter controls
    # ------------------------------------------------------------------
    "strategy_family": "continuation_only",
    "strategy_filter": STRATEGY_FILTER_MODE if USE_STRATEGY_FILTER else "off",
    "engine_mode": "intelligent" if USE_STRATEGY_FILTER else "manual",
    "enable_regime_router": USE_STRATEGY_FILTER,
    "regime_lookback_minutes": 20,

    # ------------------------------------------------------------------
    # Future router-blocked override controls
    # These are intentionally inactive placeholders for future research.
    # They should not affect current results until wired explicitly.
    # ------------------------------------------------------------------
    "allow_blocked_s_tier_with_extra_quality": ALLOW_BLOCKED_S_TIER_WITH_EXTRA_QUALITY,
    "allow_blocked_dynamic_s_tier_with_extra_quality": ALLOW_BLOCKED_DYNAMIC_S_TIER_WITH_EXTRA_QUALITY,
    "allow_blocked_a_tier_with_extra_quality": ALLOW_BLOCKED_A_TIER_WITH_EXTRA_QUALITY,
    "allow_blocked_delayed_pullback_with_extra_quality": ALLOW_BLOCKED_DELAYED_PULLBACK_WITH_EXTRA_QUALITY,

    # ------------------------------------------------------------------
    # Setup-specific quality filters
    # ------------------------------------------------------------------
    "enable_s_tier_clean_state_filter": USE_S_TIER_CLEAN_STATE_FILTER,

    "s_tier_use_entry_red_shift_floor": USE_RED_SHIFT_FLOOR,
    "a_tier_use_entry_red_shift_floor": USE_RED_SHIFT_FLOOR,

    "enable_v2_trend_health_filter": USE_TREND_HEALTH_FILTER,
    "enable_v2_continuation_health": USE_TREND_HEALTH_FILTER,
    "enable_v2_reclaim_recovery_health": USE_TREND_HEALTH_FILTER,

    "use_candle_quality_filter": USE_CANDLE_QUALITY_FILTER,
    "use_extension_filter": USE_EXTENSION_FILTER,

    # ------------------------------------------------------------------
    # Entry logic
    # ------------------------------------------------------------------
    "entry_timing": "next_bar_open",

    # ------------------------------------------------------------------
    # Fixed Nasdaq point trade levels
    # ------------------------------------------------------------------
    "sl_points": SL_POINTS,
    "tp_points": ORIGINAL_TP_POINTS,
    "be_trigger_points": BE_TRIGGER_POINTS,

    "normal_sl_points": SL_POINTS,
    "normal_tp_points": ORIGINAL_TP_POINTS,
    "normal_be_trigger_points": BE_TRIGGER_POINTS,

    # ------------------------------------------------------------------
    # Runner / exit manager controls
    # ------------------------------------------------------------------
    "enable_v5_exit_manager": True,
    "enable_runner_trailing": RUNNER_MODE != "off",
    "runner_mode": RUNNER_MODE,
    "runner_target_r": RUNNER_TARGET_R,
    "runner_trail_rules_r": RUNNER_TRAIL_RULES_R,

    "enable_be_plus_buffer": True,
    "be_plus_buffer_points": 3.0,

    "enable_legacy_early_runner_lock": USE_LEGACY_EARLY_RUNNER_LOCK,
    "legacy_early_runner_lock_only_when_tp_points_above": 60.0,
    "legacy_runner_reference_sl_points": 29.0,
    "legacy_runner_lock_trigger_r": 2.7,
    "legacy_runner_lock_r": 2.0,

    "runner_tp_points": SL_POINTS * RUNNER_TARGET_R,
    "trail_be_trigger_points": SL_POINTS * 1.0,
    "trail_lock_2r_trigger_points": SL_POINTS * 2.7,
    "trail_lock_3r_trigger_points": SL_POINTS * 3.7,
    "trail_lock_4r_trigger_points": SL_POINTS * 4.7,
    "trail_5r_tp_points": SL_POINTS * 5.0,

    "trail_lock_2r_points": SL_POINTS * 2.0,
    "trail_lock_3r_points": SL_POINTS * 3.0,
    "trail_lock_4r_points": SL_POINTS * 4.0,

    # ------------------------------------------------------------------
    # Protected continuation add-ons
    # ------------------------------------------------------------------
    "addon_requires_existing_trade_protected": ADDON_REQUIRES_EXISTING_TRADE_PROTECTED,
    "addon_requires_same_direction": ADDON_REQUIRES_SAME_DIRECTION,
    "max_open_continuation_trades": MAX_OPEN_CONTINUATION_TRADES,
    "allow_mr_addon_entries": False,
    "mr_one_trade_at_a_time": True,
    "addon_allowed_entry_families": [
        "S_TIER",
        "A_TIER",
        "DELAYED_PULLBACK",
    ],

    # ------------------------------------------------------------------
    # Mean-reversion module
    # ------------------------------------------------------------------
    "mr_target_mode": MR_TARGET_MODE,
    "mr_sl_mode": MR_STOP_MODE,
    "mr_max_sl_points": MR_MAX_SL_POINTS,
    "mr_min_target_rr": MR_MIN_TARGET_RR,

    "enable_mr_green_be": ENABLE_MR_GREEN_BE,
    "mr_be_trigger_mode": MR_BE_TRIGGER_MODE,
    "mr_be_requires_close_beyond_green": MR_BE_REQUIRES_CLOSE_BEYOND_GREEN,

    "mr_orange_confirm_closes": 2,
    "mr_orange_touch_lookback_bars": 3,
    "mr_use_locked_orange_touch": True,
    "mr_block_if_strong_continuation": True,
    "mr_require_weak_or_compressing_trend": True,
    "mr_max_entry_extension_from_orange_points": 10.0,
    "mr_first_trigger_only": True,
    "mr_cooldown_bars": 10,
    "mr_sl_cooldown_bars": 20,
    "mr_min_orange_to_vwap_distance_points": 12.0,
    "mr_require_stable_or_contracting_bands": True,
    "mr_block_if_orange_band_expanding_fast": True,
    "mr_expansion_lookback_bars": 3,
    "mr_max_orange_width_expansion_points": 8.0,
    "mr_stable_band_tolerance_points": 2.0,
    "mr_entry_priority": "after_continuation",

    # ------------------------------------------------------------------
    # Risk controls
    # ------------------------------------------------------------------
    "risk_per_trade_pct": 1.0,
    "max_daily_loss_r": MAX_DAILY_LOSS_R,
    "daily_max_consecutive_losses": MAX_CONSECUTIVE_SL,
    "daily_profit_cap_pct": DAILY_PROFIT_CAP_PCT,
    "max_consecutive_sl": MAX_CONSECUTIVE_SL,
    "max_no_tp_run_r": -6,

    # ------------------------------------------------------------------
    # Session controls
    # ------------------------------------------------------------------
    "enable_session_filter": SESSION_FILTER_ENABLED,
    "enabled_sessions": ["asia", "london", "new_york"],
    "session_windows": {
        "asia": ("00:00", "07:00"),
        "london": ("07:00", "13:30"),
        "new_york": ("13:30", "21:00"),
    },
    "allow_out_of_session_trades": False,

    # ------------------------------------------------------------------
    # Candle quality filters
    # ------------------------------------------------------------------
    "min_body_ratio": 0.25,
    "min_close_through_green": 1.0,
    "max_extension_from_green": 8.0,

    # ------------------------------------------------------------------
    # Feature lookbacks
    # ------------------------------------------------------------------
    "shift_lookback": 3,
    "acceptance_lookback": 3,
    "trend_lane_lookback": 5,
    "trend_damage_lookback": 5,
    "compression_lookback": 5,
    "flat_vwap_lookback": 5,
    "vwap_cross_lookback": 8,

    # ------------------------------------------------------------------
    # Reporting / audits / exports
    # ------------------------------------------------------------------
    "enable_v5_raw_signal_audit": ENABLE_RAW_SIGNAL_AUDIT,
    "enable_strategy_filter_block_audit": ENABLE_STRATEGY_FILTER_BLOCK_AUDIT,
    "enable_trade_attribution_report": ENABLE_TRADE_ATTRIBUTION_REPORT,
    "enable_risk_diagnostics": ENABLE_RISK_DIAGNOSTICS,
    "enable_account_survival_report": ENABLE_ACCOUNT_SURVIVAL_REPORT,
    "enable_experiment_logging": ENABLE_EXPERIMENT_LOGGING,

    "save_trade_log": True,
    "save_daily_summary": True,
    "save_skipped_signals": True,
    "save_config_snapshot": True,
    "raw_signal_audit_max_rows": 300,
    "raw_signal_audit_save_csv": True,
    "strategy_filter_block_audit_save_csv": True,
    "strategy_filter_block_audit_max_rows": 300,
}

print("Strategy control panel mapped into internal CONFIG:")
pprint(CONFIG)

Strategy control panel mapped into internal CONFIG:
{'a_tier_use_entry_red_shift_floor': False,
 'acceptance_lookback': 3,
 'addon_allowed_entry_families': ['S_TIER', 'A_TIER', 'DELAYED_PULLBACK'],
 'addon_requires_existing_trade_protected': True,
 'addon_requires_same_direction': True,
 'allow_blocked_a_tier_with_extra_quality': False,
 'allow_blocked_delayed_pullback_with_extra_quality': False,
 'allow_blocked_dynamic_s_tier_with_extra_quality': False,
 'allow_blocked_s_tier_with_extra_quality': False,
 'allow_longs': True,
 'allow_mr_addon_entries': False,
 'allow_out_of_session_trades': False,
 'allow_shorts': True,
 'be_plus_buffer_points': 3.0,
 'be_trigger_points': 36.0,
 'compression_lookback': 5,
 'csv_file': 'US100_cash_M1_NY_session_1y.csv',
 'daily_max_consecutive_losses': 2,
 'daily_profit_cap_pct': 999.0,
 'dataset_name': 'US100_cash_M1_NY_session_1y',
 'enable_a_tier_delayed_pullback_entry': True,
 'enable_account_survival_report': True,
 'enable_be_plus_buffer': True,
 

## Dataset configuration

This section defines dataset paths and dataset metadata only.

No strategy logic belongs in this section.

In [8]:
DATASETS = [
    {
        "label": "30d",
        "csv_file": "US100_cash_M1_NY_session_30d.csv",
        "dataset_name": "US100_cash_M1_NY_session_30d",
    },
    {
        "label": "1Y",
        "csv_file": "US100_cash_M1_NY_session_1y.csv",
        "dataset_name": "US100_cash_M1_NY_session_1y",
    },
    {
        "label": "2021 partial",
        "csv_file": "US100_cash_M1_NY_session_calm_2021_partial.csv",
        "dataset_name": "US100_cash_M1_NY_session_calm_2021_partial",
    },
    {
        "label": "2022 volatile",
        "csv_file": "US100_cash_M1_NY_session_volatile_2022.csv",
        "dataset_name": "US100_cash_M1_NY_session_volatile_2022",
    },
    {
        "label": "War 22/23",
        "csv_file": "US100_cash_M1_NY_session_ukraine_war_2022_2023.csv",
        "dataset_name": "US100_cash_M1_NY_session_ukraine_war_2022_2023",
    },
    {
        "label": "Recent",
        "csv_file": "US100_cash_M1_NY_session_recent_2025_2026.csv",
        "dataset_name": "US100_cash_M1_NY_session_recent_2025_2026",
    },
    {
        "label": "2023 full",
        "csv_file": "us100_cash_M1_new_york_2023_01_01_to_2024_01_01.csv",
        "dataset_name": "US100_cash_M1_NY_session_2023_full",
    },
    {
        "label": "2024 full",
        "csv_file": "us100_cash_M1_new_york_2024_01_01_to_2025_01_01.csv",
        "dataset_name": "US100_cash_M1_NY_session_2024_full",
    },
]

CONFIG["comparison_datasets"] = [
    {
        "csv_file": dataset["csv_file"],
        "dataset_name": dataset["dataset_name"],
    }
    for dataset in DATASETS
]

dataset_table = pd.DataFrame(DATASETS)
dataset_table["path"] = dataset_table["csv_file"].apply(lambda name: HISTORICAL_DATA_DIR / name)

dataset_table

,label,csv_file,dataset_name,path
0,30d,US100_cash_M1_NY_session_30d.csv,US100_cash_M1_NY_session_30d,C:\GitHub Projects\VWAP-probability-band-engin...
1,1Y,US100_cash_M1_NY_session_1y.csv,US100_cash_M1_NY_session_1y,C:\GitHub Projects\VWAP-probability-band-engin...
2,2021 partial,US100_cash_M1_NY_session_calm_2021_partial.csv,US100_cash_M1_NY_session_calm_2021_partial,C:\GitHub Projects\VWAP-probability-band-engin...
3,2022 volatile,US100_cash_M1_NY_session_volatile_2022.csv,US100_cash_M1_NY_session_volatile_2022,C:\GitHub Projects\VWAP-probability-band-engin...
4,War 22/23,US100_cash_M1_NY_session_ukraine_war_2022_2023...,US100_cash_M1_NY_session_ukraine_war_2022_2023,C:\GitHub Projects\VWAP-probability-band-engin...
5,Recent,US100_cash_M1_NY_session_recent_2025_2026.csv,US100_cash_M1_NY_session_recent_2025_2026,C:\GitHub Projects\VWAP-probability-band-engin...
6,2023 full,us100_cash_M1_new_york_2023_01_01_to_2024_01_0...,US100_cash_M1_NY_session_2023_full,C:\GitHub Projects\VWAP-probability-band-engin...
7,2024 full,us100_cash_M1_new_york_2024_01_01_to_2025_01_0...,US100_cash_M1_NY_session_2024_full,C:\GitHub Projects\VWAP-probability-band-engin...


### Market data loading helpers

This cell loads and standardises raw OHLC data.

It only prepares market data for the VWAP/sigma engine.
It does not create signals, filter trades, or simulate results.

In [9]:
REQUIRED_RAW_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
]

COLUMN_ALIASES = {
    "datetime": [
        "datetime",
        "time",
        "timestamp",
        "date",
        "Date",
        "Time",
        "Datetime",
        "Local time",
        "Gmt time",
        "GMT time",
    ],
    "open": ["open", "Open", "OPEN"],
    "high": ["high", "High", "HIGH"],
    "low": ["low", "Low", "LOW"],
    "close": ["close", "Close", "CLOSE"],
    "tick_volume": [
        "tick_volume",
        "volume",
        "Volume",
        "tickvol",
        "Tick Volume",
        "Tick volume",
    ],
    "spread": ["spread", "Spread"],
    "real_volume": ["real_volume", "Real Volume", "real volume"],
}


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    Return the first matching column from a list of possible column names.
    """
    existing = set(df.columns)

    for col in candidates:
        if col in existing:
            return col

    lower_map = {str(col).lower(): col for col in df.columns}

    for col in candidates:
        if str(col).lower() in lower_map:
            return lower_map[str(col).lower()]

    return None


def standardise_raw_ohlc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename common OHLC column names into the standard names used by this notebook.
    """
    out = df.copy()
    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        matched_col = find_column(out, aliases)

        if matched_col is not None and matched_col != standard_name:
            rename_map[matched_col] = standard_name

    out = out.rename(columns=rename_map)

    return out


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """
    Validate that the dataframe contains the required columns.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )


def prepare_raw_ohlc_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw OHLC data for the automated strategy research layer.

    This function only cleans and validates raw market data.

    It does not calculate VWAP.
    It does not calculate probability bands.
    It does not change the existing VWAP engine logic.
    """
    out = standardise_raw_ohlc_columns(df)

    validate_required_columns(out, REQUIRED_RAW_COLUMNS)

    out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
    out = out.dropna(subset=["datetime"]).copy()

    numeric_cols = ["open", "high", "low", "close"]

    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).copy()

    for optional_col in ["tick_volume", "spread", "real_volume"]:
        if optional_col in out.columns:
            out[optional_col] = pd.to_numeric(out[optional_col], errors="coerce")

    out = out.sort_values("datetime").reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    return out


def list_candidate_data_files() -> list[Path]:
    """
    Find likely CSV or Parquet files inside the project.
    """
    patterns = ["*.csv", "*.parquet"]
    ignored_parts = {
        ".git",
        ".venv",
        "venv",
        "__pycache__",
        ".ipynb_checkpoints",
    }

    files = []

    for pattern in patterns:
        files.extend(PROJECT_ROOT.rglob(pattern))

    clean_files = [
        file for file in files
        if not any(part in ignored_parts for part in file.parts)
    ]

    return sorted(set(clean_files))


def resolve_data_file(config: dict) -> Path:
    """
    Resolve the configured data file.

    The config can use:
    - a file name, e.g. US100_cash_M1_NY_session_30d.csv
    - a relative path, e.g. data/historical/file.csv
    - an absolute path
    """
    configured_file = Path(config["csv_file"])

    if configured_file.is_absolute() and configured_file.exists():
        return configured_file

    direct_project_path = PROJECT_ROOT / configured_file

    if direct_project_path.exists():
        return direct_project_path

    candidate_files = list_candidate_data_files()

    matching_files = [
        file for file in candidate_files
        if file.name == configured_file.name
    ]

    if len(matching_files) == 1:
        return matching_files[0]

    if len(matching_files) > 1:
        print("Multiple matching files found:")

        for file in matching_files:
            print("-", file.relative_to(PROJECT_ROOT))

        raise ValueError(
            f"Multiple files named {configured_file.name} were found. "
            "Use a more specific relative path in CONFIG['csv_file']."
        )

    print(f"Could not find configured file: {config['csv_file']}")
    print("\nAvailable candidate files:")

    for file in candidate_files[:100]:
        print("-", file.relative_to(PROJECT_ROOT))

    raise FileNotFoundError(
        f"Could not find configured data file: {config['csv_file']}"
    )


def load_market_data(config: dict) -> tuple[pd.DataFrame, Path]:
    """
    Load the configured CSV or Parquet file and return a cleaned OHLC dataframe.
    """
    data_file = resolve_data_file(config)

    if data_file.suffix.lower() == ".csv":
        raw_df = pd.read_csv(data_file)
    elif data_file.suffix.lower() == ".parquet":
        raw_df = pd.read_parquet(data_file)
    else:
        raise ValueError(f"Unsupported file type: {data_file.suffix}")

    prepared_df = prepare_raw_ohlc_dataframe(raw_df)

    return prepared_df, data_file

## VWAP and sigma band engine

This section calculates:
- VWAP
- EWMA sigma
- green/orange/red bands
- z-score
- zone classification

In [10]:
import importlib


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import src.config as engine_config_module
importlib.reload(engine_config_module)

from src.config import CONFIG as ENGINE_CONFIG
from src.reference import compute_reference
from src.sigma import compute_sigma, compute_bands
from src.zones import compute_zscore, classify_zones_series


print("Loaded existing VWAP engine modules.")
print("Engine config snapshot:")
print(f"- reference_type: {ENGINE_CONFIG.get('reference_type')}")
print(f"- vol_method: {ENGINE_CONFIG.get('vol_method')}")
print(f"- zone_thresholds: {ENGINE_CONFIG.get('zone_thresholds')}")

Loaded existing VWAP engine modules.
Engine config snapshot:
- reference_type: VWAP
- vol_method: ewma
- zone_thresholds: [0.5, 1.0, 2.0]


In [11]:
REQUIRED_ENGINE_OUTPUT_COLUMNS = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
]


ENGINE_COLUMN_ALIASES = {
    "vwap": ["vwap", "VWAP", "reference", "ref", "reference_line"],
    "upper_green": ["upper_green", "upper_1", "upper_band_1", "band_1p", "band_1_plus", "band_1+", "z1_upper", "upper_sigma_1"],
    "upper_orange": ["upper_orange", "upper_2", "upper_band_2", "band_2p", "band_2_plus", "band_2+", "z2_upper", "upper_sigma_2"],
    "upper_red": ["upper_red", "upper_3", "upper_band_3", "band_3p", "band_3_plus", "band_3+", "z3_upper", "upper_sigma_3"],
    "lower_green": ["lower_green", "lower_1", "lower_band_1", "band_1n", "band_1m", "band_1_minus", "band_1-", "z1_lower", "lower_sigma_1"],
    "lower_orange": ["lower_orange", "lower_2", "lower_band_2", "band_2n", "band_2m", "band_2_minus", "band_2-", "z2_lower", "lower_sigma_2"],
    "lower_red": ["lower_red", "lower_3", "lower_band_3", "band_3n", "band_3m", "band_3_minus", "band_3-", "z3_lower", "lower_sigma_3"],
}


def add_engine_band_aliases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add automation-friendly VWAP/band aliases to the existing engine output.

    This does not change model values.
    It only creates easier column names for the automated strategy layer.
    """
    out = df.copy()

    for standard_name, aliases in ENGINE_COLUMN_ALIASES.items():
        if standard_name in out.columns:
            continue

        matched_col = find_column(out, aliases)

        if matched_col is not None:
            out[standard_name] = out[matched_col]

    return out


def validate_engine_output_columns(df: pd.DataFrame) -> None:
    """
    Confirm that the engine output contains the columns needed by the automated strategy layer.
    """
    missing = [col for col in REQUIRED_ENGINE_OUTPUT_COLUMNS if col not in df.columns]

    if missing:
        raise ValueError(
            "Missing required engine output columns: "
            + ", ".join(missing)
            + "\n\nAvailable columns:\n"
            + ", ".join(map(str, df.columns))
        )

    print("All required engine output columns are available.")


def build_existing_engine_output(raw_ohlc_df: pd.DataFrame, engine_config: dict) -> pd.DataFrame:
    """
    Build VWAP probability-band output from raw OHLC data using the existing project logic.

    This function does not modify the existing model.

    It calls the existing source functions and then adds automation-friendly aliases.
    """
    df = raw_ohlc_df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    if "tick_volume" not in df.columns:
        df["tick_volume"] = 1.0

    df["tick_volume"] = (
        pd.to_numeric(df["tick_volume"], errors="coerce")
        .fillna(1.0)
        .clip(lower=1.0)
    )

    df["typical_price"] = (df["high"] + df["low"] + df["close"]) / 3.0
    df["session_date"] = df["datetime"].dt.date

    df["reference"] = compute_reference(df, engine_config)
    df["price_deviation"] = df["close"] - df["reference"]

    df["sigma"] = compute_sigma(df, engine_config)

    bands = compute_bands(df, df["sigma"])
    df = pd.concat([df, bands], axis=1)

    df["z_score"] = compute_zscore(df)
    df["zone"] = classify_zones_series(df["z_score"], engine_config["zone_thresholds"])

    df = add_engine_band_aliases(df)

    df["candle_range"] = df["high"] - df["low"]
    df["candle_body"] = (df["close"] - df["open"]).abs()
    df["body_ratio"] = np.where(
        df["candle_range"] > 0,
        df["candle_body"] / df["candle_range"],
        0.0,
    )

    validate_engine_output_columns(df)

    return df

In [12]:
def prepare_automation_dataframe(engine_df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare VWAP engine output for the automated strategy layer.

    This function validates and cleans the columns required for later
    signal generation.
    """
    out = engine_df.copy()

    out = add_engine_band_aliases(out)
    validate_engine_output_columns(out)

    out["datetime"] = pd.to_datetime(out["datetime"], utc=True, errors="coerce")
    out = out.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "vwap",
        "upper_green",
        "upper_orange",
        "upper_red",
        "lower_green",
        "lower_orange",
        "lower_red",
    ]

    optional_numeric_cols = [
        "sigma",
        "z_score",
        "tick_volume",
        "spread",
        "real_volume",
    ]

    for col in numeric_cols + optional_numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.dropna(subset=numeric_cols).reset_index(drop=True)

    out["candle_range"] = out["high"] - out["low"]
    out["candle_body"] = (out["close"] - out["open"]).abs()
    out["body_ratio"] = np.where(
        out["candle_range"] > 0,
        out["candle_body"] / out["candle_range"],
        0.0,
    )

    out["bar_index"] = np.arange(len(out))

    return out

In [13]:
def summarise_engine_output(df: pd.DataFrame, config: dict, engine_config: dict) -> dict:
    """
    Create a compact summary of the VWAP/sigma engine output.
    """
    summary = {
        "dataset_name": config["dataset_name"],
        "strategy_version": config["strategy_version"],
        "rows": len(df),
        "start_datetime": df["datetime"].min(),
        "end_datetime": df["datetime"].max(),
        "reference_type": engine_config.get("reference_type"),
        "vol_method": engine_config.get("vol_method"),
        "zone_thresholds": engine_config.get("zone_thresholds"),
        "mean_sigma": df["sigma"].mean() if "sigma" in df.columns else None,
        "median_sigma": df["sigma"].median() if "sigma" in df.columns else None,
        "mean_green_band_width": (df["upper_green"] - df["lower_green"]).mean(),
        "median_green_band_width": (df["upper_green"] - df["lower_green"]).median(),
        "mean_red_band_width": (df["upper_red"] - df["lower_red"]).mean(),
        "median_red_band_width": (df["upper_red"] - df["lower_red"]).median(),
        "zone_counts": df["zone"].value_counts(dropna=False).to_dict() if "zone" in df.columns else None,
        "missing_vwap_values": int(df["vwap"].isna().sum()),
        "missing_upper_green_values": int(df["upper_green"].isna().sum()),
        "missing_lower_green_values": int(df["lower_green"].isna().sum()),
    }

    return summary

In [14]:
raw_ohlc_df, DATA_FILE = load_market_data(CONFIG)

print(f"Loaded market data from: {DATA_FILE}")
print(f"Rows loaded: {len(raw_ohlc_df):,}")

raw_ohlc_df.tail()

Loaded market data from: C:\GitHub Projects\VWAP-probability-band-engine\data\historical\US100_cash_M1_NY_session_1y.csv
Rows loaded: 51,666


,datetime,open,high,low,close,tick_volume,spread,real_volume,candle_range,candle_body,body_ratio
51661,2026-05-15 16:25:00,29113.95,29142.55,29113.95,29137.95,492,190,0,28.6,24.0,0.839161
51662,2026-05-15 16:26:00,29138.25,29143.05,29130.75,29138.25,417,190,0,12.3,0.0,0.000000
51663,2026-05-15 16:27:00,29138.05,29141.25,29121.05,29123.05,461,190,0,20.2,15.0,0.742574
51664,2026-05-15 16:28:00,29123.25,29163.45,29123.25,29163.05,506,190,0,40.2,39.8,0.990050
51665,2026-05-15 16:29:00,29163.45,29185.35,29162.35,29179.55,515,190,0,23.0,16.1,0.700000


In [15]:
engine_df = build_existing_engine_output(raw_ohlc_df, ENGINE_CONFIG)

print(f"Engine output rows: {len(engine_df):,}")
print(f"Engine output columns: {len(engine_df.columns):,}")

engine_preview_cols = [
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "z_score",
    "zone",
]

engine_df[engine_preview_cols].tail(10)

✅ VWAP computed — resets per session: True
✅ Sigma computed (ewma)
   Mean sigma : 20.872847
   Min sigma  : 6.269622
   Max sigma  : 332.844379
   Floor used : 6.269622
All required engine output columns are available.
Engine output rows: 51,666
Engine output columns: 31


,datetime,close,vwap,upper_green,upper_orange,upper_red,lower_green,lower_orange,lower_red,z_score,zone
51656,2026-05-15 16:20:00+00:00,29110.25,29147.849531,29170.459877,29193.070224,29215.680570,29125.239184,29102.628838,29080.018491,-1.662935,Z2-
51657,2026-05-15 16:21:00+00:00,29104.75,29147.626312,29170.757618,29193.888924,29217.020230,29124.495006,29101.363700,29078.232394,-1.853605,Z2-
51658,2026-05-15 16:22:00+00:00,29100.75,29147.382730,29170.705873,29194.029016,29217.352160,29124.059587,29100.736443,29077.413300,-1.999419,Z2-
51659,2026-05-15 16:23:00+00:00,29103.45,29147.157512,29170.751702,29194.345892,29217.940082,29123.563323,29099.969133,29076.374943,-1.852469,Z2-
51660,2026-05-15 16:24:00+00:00,29108.45,29146.937074,29171.010477,29195.083881,29219.157284,29122.863670,29098.790266,29074.716863,-1.598738,Z2-
51661,2026-05-15 16:25:00+00:00,29137.95,29146.822099,29173.619868,29200.417636,29227.215404,29120.024331,29093.226563,29066.428795,-0.331076,Z0
51662,2026-05-15 16:26:00+00:00,29138.25,29146.762745,29175.706915,29204.651085,29233.595255,29117.818575,29088.874405,29059.930235,-0.294109,Z0
51663,2026-05-15 16:27:00+00:00,29123.05,29146.636757,29176.200764,29205.764772,29235.328779,29117.072750,29087.508743,29057.944736,-0.797820,Z1-
51664,2026-05-15 16:28:00+00:00,29163.05,29146.661339,29179.988687,29213.316034,29246.643381,29113.333992,29080.006645,29046.679298,0.491748,Z0
51665,2026-05-15 16:29:00+00:00,29179.55,29146.881549,29184.730812,29222.580074,29260.429336,29109.032287,29071.183025,29033.333763,0.863120,Z1+


In [16]:
automation_ready_df = prepare_automation_dataframe(engine_df)

print(f"Automation-ready dataframe rows: {len(automation_ready_df):,}")
print(f"Automation-ready dataframe columns: {len(automation_ready_df.columns):,}")

automation_preview_cols = [
    "bar_index",
    "datetime",
    "close",
    "vwap",
    "upper_green",
    "upper_orange",
    "upper_red",
    "lower_green",
    "lower_orange",
    "lower_red",
    "body_ratio",
]

automation_ready_df[automation_preview_cols].tail(10)

All required engine output columns are available.
Automation-ready dataframe rows: 51,666
Automation-ready dataframe columns: 32


,bar_index,datetime,close,vwap,upper_green,upper_orange,upper_red,lower_green,lower_orange,lower_red,body_ratio
51656,51656,2026-05-15 16:20:00+00:00,29110.25,29147.849531,29170.459877,29193.070224,29215.680570,29125.239184,29102.628838,29080.018491,0.869919
51657,51657,2026-05-15 16:21:00+00:00,29104.75,29147.626312,29170.757618,29193.888924,29217.020230,29124.495006,29101.363700,29078.232394,0.491228
51658,51658,2026-05-15 16:22:00+00:00,29100.75,29147.382730,29170.705873,29194.029016,29217.352160,29124.059587,29100.736443,29077.413300,0.225000
51659,51659,2026-05-15 16:23:00+00:00,29103.45,29147.157512,29170.751702,29194.345892,29217.940082,29123.563323,29099.969133,29076.374943,0.258929
51660,51660,2026-05-15 16:24:00+00:00,29108.45,29146.937074,29171.010477,29195.083881,29219.157284,29122.863670,29098.790266,29074.716863,0.456140
51661,51661,2026-05-15 16:25:00+00:00,29137.95,29146.822099,29173.619868,29200.417636,29227.215404,29120.024331,29093.226563,29066.428795,0.839161
51662,51662,2026-05-15 16:26:00+00:00,29138.25,29146.762745,29175.706915,29204.651085,29233.595255,29117.818575,29088.874405,29059.930235,0.000000
51663,51663,2026-05-15 16:27:00+00:00,29123.05,29146.636757,29176.200764,29205.764772,29235.328779,29117.072750,29087.508743,29057.944736,0.742574
51664,51664,2026-05-15 16:28:00+00:00,29163.05,29146.661339,29179.988687,29213.316034,29246.643381,29113.333992,29080.006645,29046.679298,0.990050
51665,51665,2026-05-15 16:29:00+00:00,29179.55,29146.881549,29184.730812,29222.580074,29260.429336,29109.032287,29071.183025,29033.333763,0.700000


In [17]:
engine_summary = summarise_engine_output(automation_ready_df, CONFIG, ENGINE_CONFIG)

pprint(engine_summary)

{'dataset_name': 'US100_cash_M1_NY_session_1y',
 'end_datetime': Timestamp('2026-05-15 16:29:00+0000', tz='UTC'),
 'mean_green_band_width': np.float64(41.74569497080791),
 'mean_red_band_width': np.float64(125.23708491242446),
 'mean_sigma': np.float64(20.872847485404108),
 'median_green_band_width': np.float64(29.330398258938658),
 'median_red_band_width': np.float64(87.99119477681234),
 'median_sigma': np.float64(14.665199129468522),
 'missing_lower_green_values': 0,
 'missing_upper_green_values': 0,
 'missing_vwap_values': 0,
 'reference_type': 'VWAP',
 'rows': 51666,
 'start_datetime': Timestamp('2025-05-16 13:10:00+0000', tz='UTC'),
 'strategy_version': 'algorithmic_vwap_sigma_engine',
 'vol_method': 'ewma',
 'zone_counts': {'Z0': 15725,
                 'Z1+': 4819,
                 'Z1-': 4641,
                 'Z2+': 7488,
                 'Z2-': 6442,
                 'Z3+': 7273,
                 'Z3-': 5278},
 'zone_thresholds': [0.5, 1.0, 2.0]}


## Continuation signal engine

This section creates raw continuation setup signals:
- S-tier
- dynamic S-tier
- A-tier
- delayed pullback

## Mean-reversion signal engine

This section creates optional mean-reversion signals.

Mean reversion is separate from continuation.
It targets VWAP and uses structure stops.
It is disabled by default.

## Router and strategy filter

This section decides whether raw signals are allowed through.

The router/filter is a permission layer.
It should not be confused with raw signal generation.

The intended research flow is:

```text
raw setup detected
entry family identified
router permission checked
quality filters checked
final signal selected

## Trade simulation and risk engine

This section simulates trades from final selected signals.

It handles:
- entry price
- stop loss and target
- breakeven
- runner trailing
- daily risk gates
- protected continuation add-ons
- open-trade blocking

## Experiment runner

This section runs the selected strategy configuration across all datasets.

The runner should store results cleanly without displaying every audit immediately.

## Quick results

Fast summary of the current configuration.

This section should show the main comparison table and headline results before any detailed audits.

## Core charts

Main visual outputs for the current strategy run.

## Audits and detailed diagnostics

These sections explain why the model did or did not trade.

They are detailed diagnostics, not the first results the user should see.

### Raw signal audit

### Strategy-filter block audit

### Trade attribution report

### Mean-reversion diagnostics

### Protected add-on diagnostics

## Experiment logging

This section saves the current run configuration and headline results.

## Future live runner notes

The future live file should use this notebook's final clean config structure.

Live runner goals:
- one file
- signal-only by default
- optional order placement later
- use same MT5 account/feed/symbol as overlay
- same lookback/session/VWAP/sigma settings as overlay
- clear logging of raw signal, final signal, and skip reason